# Preparations

In [1]:
#@title Imports
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [2]:
#@title Datasets import

os.makedirs('processed_data', exist_ok=True)

base_url = 'https://raw.githubusercontent.com/DheandraKhairunnisaP/DataScience-ProjectUAS/refs/heads/main/data/raw'
sub_dir = f'{base_url}/Crop%20Yield%20Prediction%20Dataset'

file_urls = {
    'pesticides.csv': f'{sub_dir}/pesticides.csv',
    'rainfall.csv': f'{sub_dir}/rainfall.csv',
    'temp.csv': f'{sub_dir}/temp.csv',
    'yield.csv': f'{sub_dir}/yield.csv',
    'yield_df.csv': f'{sub_dir}/yield_df.csv',
    'crop_yield.csv': f'{base_url}/crop_yield.csv',
    'crop_yield_dataset.csv': f'{base_url}/crop_yield_dataset.csv'
}

datasets = {
    name: pd.read_csv(url, low_memory=False)
    for name, url in file_urls.items()
}

# Analisis Kondisi Data

In [3]:
#@title Konfigurasi Pemeriksaan Fitur dan Target
rainfall_col = 'average_rain_fall_mm_per_year'

outlier_features = {
    'pesticides.csv': ['Value'],
    'rainfall.csv': [rainfall_col],
    'temp.csv': ['avg_temp'],
    'yield.csv': ['Value'],
    'yield_df.csv': ['hg/ha_yield', 'pesticides_tonnes', 'avg_temp'],
    'crop_yield.csv': ['Yield_tons_per_hectare'],
    'crop_yield_dataset.csv': ['Yield_ton_per_ha']
}

targets = {
    'yield.csv': 'Value',
    'yield_df.csv': 'hg/ha_yield',
    'crop_yield.csv': 'Yield_tons_per_hectare',
    'crop_yield_dataset.csv': 'Yield_ton_per_ha'
}

In [4]:
#@title Pemeriksaan Missing Values, Duplikat, Outlier, dan Nilai Negatif
print("RINGKASAN KONDISI DATA")
print("-" * 75)

for nama, df in datasets.items():
    temp_df = df.copy()
    if nama == 'rainfall.csv':
        temp_df[rainfall_col] = pd.to_numeric(temp_df[rainfall_col], errors='coerce')

    missing = temp_df.isnull().sum().sum()

    if nama == 'yield_df.csv':
        duplicate = temp_df.drop(columns=['Unnamed: 0']).duplicated().sum()
    else:
        duplicate = temp_df.duplicated().sum()

    total_outlier = 0
    for col in outlier_features[nama]:
        data = pd.to_numeric(temp_df[col], errors='coerce').dropna()
        q1 = data.quantile(0.25)
        q3 = data.quantile(0.75)
        iqr = q3 - q1
        batas_bawah = q1 - 1.5 * iqr
        batas_atas = q3 + 1.5 * iqr
        total_outlier += ((data < batas_bawah) | (data > batas_atas)).sum()

    negatif = 0
    if nama in targets:
        negatif = (pd.to_numeric(temp_df[targets[nama]], errors='coerce') < 0).sum()

    print(
        f"{nama:25} | "
        f"Missing: {missing:5} | "
        f"Duplikat: {duplicate:5} | "
        f"Outlier: {total_outlier:5} | "
        f"Negatif: {negatif:4}"
    )

RINGKASAN KONDISI DATA
---------------------------------------------------------------------------
pesticides.csv            | Missing:     0 | Duplikat:     0 | Outlier:   615 | Negatif:    0
rainfall.csv              | Missing:   780 | Duplikat:     0 | Outlier:     0 | Negatif:    0
temp.csv                  | Missing:  2547 | Duplikat:  6958 | Outlier:     2 | Negatif:    0
yield.csv                 | Missing:     0 | Duplikat:     0 | Outlier:  3169 | Negatif:    0
yield_df.csv              | Missing:     0 | Duplikat:  2310 | Outlier:  3511 | Negatif:    0
crop_yield.csv            | Missing:     0 | Duplikat:     0 | Outlier:    84 | Negatif:  231
crop_yield_dataset.csv    | Missing:  4569 | Duplikat:     0 | Outlier:     0 | Negatif:    0


In [5]:
#@title Distribusi Data Kategorikal
kategori = {
    'yield_df.csv': ['Area', 'Item'],
    'crop_yield.csv': ['Region', 'Soil_Type', 'Crop'],
    'crop_yield_dataset.csv': ['Crop', 'Region', 'Soil_Type']
}

print("DISTRIBUSI KATEGORI")
print("-" * 45)

for nama, kolom in kategori.items():
    for col in kolom:
        frekuensi = datasets[nama][col].value_counts()
        print(f"{nama} - {col}: {frekuensi.min()} sampai {frekuensi.max()}")

DISTRIBUSI KATEGORI
---------------------------------------------
yield_df.csv - Area: 24 sampai 4048
yield_df.csv - Item: 556 sampai 4276
crop_yield.csv - Region: 249699 sampai 250173
crop_yield.csv - Soil_Type: 166283 sampai 167119
crop_yield.csv - Crop: 166349 sampai 166824
crop_yield_dataset.csv - Crop: 2477 sampai 2536
crop_yield_dataset.csv - Region: 2446 sampai 2561
crop_yield_dataset.csv - Soil_Type: 3284 sampai 3388


In [6]:
#@title Deteksi Noise Data
print("INDIKASI NOISE")
print("-" * 45)

noise_count = (
    datasets['rainfall.csv'][rainfall_col]
    .astype(str)
    .str.strip() == '..'
).sum()

print("Nilai '..' pada rainfall:", noise_count)
print("Pemeriksaan selesai.")

INDIKASI NOISE
---------------------------------------------
Nilai '..' pada rainfall: 6
Pemeriksaan selesai.


# Data Preparation
Data cleaning, feature engineering, data splitting dan data transformation

In [11]:
#@title Cleaning Configuration
cleaning_rules = {
    'rainfall.csv': {
        'num_cols': ['average_rain_fall_mm_per_year'],
        'group_impute': ('average_rain_fall_mm_per_year', 'Area')
    },
    'temp.csv': {
        'num_cols': ['avg_temp'],
        'group_impute': ('avg_temp', 'country'),
        'dedup': True
    },
    'pesticides.csv': {
        'num_cols': ['Year', 'Value']
    },
    'yield.csv': {
        'num_cols': ['Year', 'Value']
    },
    'yield_df.csv': {
        'num_cols': ['Year', 'hg/ha_yield', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp'],
        'dedup': True
    },
    'crop_yield.csv': {
        'num_cols': ['Rainfall_mm', 'Temperature_Celsius', 'Days_to_Harvest', 'Yield_tons_per_hectare'],
        'non_negative_col': 'Yield_tons_per_hectare',
        'dedup': True
    },
    'crop_yield_dataset.csv': {
        'num_cols': ['Soil_pH', 'Rainfall_mm', 'Temperature_C', 'Humidity_pct', 'Fertilizer_Used_kg', 'Pesticides_Used_kg', 'Planting_Density', 'Yield_ton_per_ha'],
        'cat_fillna': ['Irrigation', 'Previous_Crop'],
        'dedup': True
    }
}

In [12]:
#@title Data Cleaning
cleaned = {}

for nama, data in datasets.items():
    df = data.copy()
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

    rules = cleaning_rules.get(nama, {})

    for col in rules.get('num_cols', []):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    if rules.get('dedup'):
        df = df.drop_duplicates()

    if 'group_impute' in rules:
        target_col, group_col = rules['group_impute']
        df[target_col] = df[target_col].fillna(df.groupby(group_col)[target_col].transform('median'))
        df[target_col] = df[target_col].fillna(df[target_col].median())

    if 'non_negative_col' in rules:
        col = rules['non_negative_col']
        df = df[df[col] >= 0]

    for col in rules.get('cat_fillna', []):
        df[col] = df[col].fillna('Unknown')

    cleaned[nama] = df

print("HASIL DATA CLEANING")
print("-" * 65)
for nama, df in cleaned.items():
    print(f"{nama:25} | Baris: {len(df):7} | Missing: {df.isnull().sum().sum():5}")
print("\nData cleaning selesai.")

HASIL DATA CLEANING
-----------------------------------------------------------------
pesticides.csv            | Baris:    4349 | Missing:     0
rainfall.csv              | Baris:    6727 | Missing:     0
temp.csv                  | Baris:   64353 | Missing:     0
yield.csv                 | Baris:   56717 | Missing:     0
yield_df.csv              | Baris:   25932 | Missing:     0
crop_yield.csv            | Baris:  999769 | Missing:     0
crop_yield_dataset.csv    | Baris:   10000 | Missing:     0

Data cleaning selesai.


In [13]:
#@title Feature Engineering Configuration
fe_config = {
    'Dataset 1': {
        'source': 'yield_df.csv',
        'interaction': ('average_rain_fall_mm_per_year', 'avg_temp'),
        'features': ['Area', 'Item', 'Year', 'average_rain_fall_mm_per_year', 'pesticides_tonnes', 'avg_temp', 'Rainfall_Temp_Interaction'],
        'target': 'hg/ha_yield'
    },
    'Dataset 2': {
        'source': 'crop_yield.csv',
        'interaction': ('Rainfall_mm', 'Temperature_Celsius'),
        'features': ['Region', 'Soil_Type', 'Crop', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used', 'Weather_Condition', 'Days_to_Harvest', 'Rainfall_Temp_Interaction'],
        'target': 'Yield_tons_per_hectare'
    },
    'Dataset 3': {
        'source': 'crop_yield_dataset.csv',
        'interaction': ('Rainfall_mm', 'Temperature_C'),
        'features': ['Crop', 'Region', 'Soil_Type', 'Soil_pH', 'Rainfall_mm', 'Temperature_C', 'Humidity_pct', 'Fertilizer_Used_kg', 'Irrigation', 'Pesticides_Used_kg', 'Planting_Density', 'Previous_Crop', 'Rainfall_Temp_Interaction'],
        'target': 'Yield_ton_per_ha'
    }
}

In [14]:
#@title Feature Engineering
feature_data = {}
features = {}
targets = {}

print("HASIL FEATURE ENGINEERING")
print("-" * 60)

for nama, cfg in fe_config.items():
    df = cleaned[cfg['source']].copy()
    rain_col, temp_col = cfg['interaction']
    df['Rainfall_Temp_Interaction'] = df[rain_col] * df[temp_col]

    feature_data[nama] = df
    features[nama] = df[cfg['features']].copy()
    targets[nama] = df[cfg['target']].copy()

    filename = nama.lower().replace(" ", "_")
    df.to_csv(f'processed_data/{filename}_prepared.csv', index=False)

    print(f"{nama} -> Data: {len(df)} | Fitur: {features[nama].shape[1]} | Target: {cfg['target']}")

print("-" * 60)
print("Data preparation berhasil disimpan.")

HASIL FEATURE ENGINEERING
------------------------------------------------------------
Dataset 1 -> Data: 25932 | Fitur: 7 | Target: hg/ha_yield
Dataset 2 -> Data: 999769 | Fitur: 10 | Target: Yield_tons_per_hectare
Dataset 3 -> Data: 10000 | Fitur: 13 | Target: Yield_ton_per_ha
------------------------------------------------------------
Data preparation berhasil disimpan.


In [15]:
#@title Data Splitting
splits = {}

for nama in features:
    X = features[nama]
    y = targets[nama]

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42
    )

    splits[nama] = {
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test
    }

    print(f"{nama} -> Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Dataset 1 -> Train: 18152 | Val: 3890 | Test: 3890
Dataset 2 -> Train: 699838 | Val: 149965 | Test: 149966
Dataset 3 -> Train: 7000 | Val: 1500 | Test: 1500


In [16]:
#@title Data Transformation

processed_splits = {}
preprocessors = {}

for nama, data_split in splits.items():
    X_train = data_split['X_train']
    X_val = data_split['X_val']
    X_test = data_split['X_test']

    cat_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    num_cols = X_train.select_dtypes(include=['number']).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ]
    )

    X_train_proc = preprocessor.fit_transform(X_train)
    X_val_proc = preprocessor.transform(X_val)
    X_test_proc = preprocessor.transform(X_test)

    processed_splits[nama] = {
        'X_train': X_train_proc,
        'X_val': X_val_proc,
        'X_test': X_test_proc,
        'y_train': data_split['y_train'].to_numpy(),
        'y_val': data_split['y_val'].to_numpy(),
        'y_test': data_split['y_test'].to_numpy(),
        'feature_names': preprocessor.get_feature_names_out().tolist()
    }
    preprocessors[nama] = preprocessor

    print(f"{nama} -> Total fitur setelah transformasi: {X_train_proc.shape[1]}")

Dataset 1 -> Total fitur setelah transformasi: 116
Dataset 2 -> Total fitur setelah transformasi: 27
Dataset 3 -> Total fitur setelah transformasi: 28


In [17]:
#@title Simpan Dataset
for nama in processed_splits:
    prefix = nama.lower().replace(" ", "_")
    joblib.dump(processed_splits[nama], f'processed_data/{prefix}_splits.pkl')
    joblib.dump(preprocessors[nama], f'processed_data/{prefix}_preprocessor.pkl')

print("Seluruh file split dan preprocessor berhasil disimpan.")

Seluruh file split dan preprocessor berhasil disimpan.
